In [ ]:
from pathlib import Path
from glob import glob

import xarray as xr
from numpy import allclose
from pandas import DataFrame
import random

from geopy.location import Location
from geopy.geocoders import Nominatim

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import seaborn as sns

sns.set_style('whitegrid')

# Introduction

Data shared contains **storm surge annual maxima data based on the DCPP simulations**. The folder contains 8 netcdf files, containing the data from 8 different models. 

Data is stored using **xarray**, which allows metadata, labels, and coordinates to be embedded directly in the files. As a result, the dataset should be largely self-explanatory and relatively easy to explore.

At each grid point (~ coastal site), there are **three possible bias corrections**: 
- coordinate *bc_flag indicates which bias correction should be used at each grid point*. 
- coordinate *model_valid* indicates whether the data from the model should be considered reliable at a given grid point.

Below, I have included a few simple examples showing how to interrogate the xarray Dataset and extract the data you may need, for convenience.


**Background information on the DCPP simulations**
  * https://confluence.ecmwf.int/display/CKB/CMIP6%3A+Decadal+climate+predictions




**linked packages**
  * https://docs.xarray.dev/en/stable/index.html

# Settings

In [ ]:
# color palette finder https://python-graph-gallery.com/color-palette-finder/
colors = ['#7FF9F7FF', '#0B2483FF','#0F8CDEFF','#4C4CFFFF','#040509FF','#23B4DCFF','#3C7084FF']

In [ ]:
fs_title = 9.5

In [ ]:
geolocator = Nominatim(user_agent="geo_lookup")

In [ ]:
path = '../input/Annual_max_DCPP_20260112/'

In [ ]:
export_dir_figures = Path("../output/exploration")

In [ ]:
save_site_plot = True
save_time_series_plot = True

In [ ]:
ls_messages = []

# Loading data

In [ ]:
ls_files = [file for file in glob(path + '*.nc')]
ls_files

['../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc']

In [ ]:
file_example = ls_files[-2]

In [ ]:
model_name = file_example.split('/')[-1].split('Annual_max_')[-1].split('.')[0]

ls_messages.append(f'Processing model: {model_name}')
model_name

'CanESM5'

In [ ]:
ds = xr.open_dataset(file_example, engine="netcdf4")

# Data Structure

In [ ]:
ds

<xarray.Dataset> Size: 349MB
Dimensions:      (sample: 660, member: 2, bc: 3, sites: 11022)
Coordinates:
  * sample       (sample) int64 5kB 0 1 2 3 4 5 6 ... 654 655 656 657 658 659
  * member       (member) int64 16B 1 2
  * bc           (bc) int64 24B 1 2 3
  * sites        (sites) int64 88kB 0 1 2 3 4 ... 11017 11018 11019 11020 11021
    sim_year     (sample) int64 5kB ...
    lead         (sample) int64 5kB ...
    lon          (sites) float64 88kB ...
    lat          (sites) float64 88kB ...
    bc_flag      (sites) uint8 11kB ...
    model_valid  (sites) bool 11kB ...
Data variables:
    annualMax    (sample, member, bc, sites) float64 349MB ...
Attributes:
    description:  Modelled storm-surge annual maxima organized by simulated y...
    model:        CanESM5

In [ ]:
ls_messages.append(f'Dataset variables: {list(ds.data_vars)}')

for var in ds.data_vars:
    ls_messages.append(f' - {var}: {ds[var].attrs["long_name"]} [{ds[var].attrs["units"]}]')    

In [ ]:
ls_messages.append(f' - sim_year_range: {ds["sim_year"].to_series().describe().loc[["min", "max"]].astype(int).values}')

ds['sim_year'].to_series().describe().astype(int)

In [ ]:
ls_messages

In [ ]:
ds.sample.sim_year == 2020

In [ ]:
ls_messages.append(f' - locations_total: {ds["sites"].shape[0]}')

ds['sites']

In [ ]:
ds["bc_flag"].astype(int)-1 # per site

# Prepare Data 

### Bias Correction

In [ ]:
ds

In [ ]:
annualMax_pref = xr.apply_ufunc(
    lambda v, i: v[i],
    ds["annualMax"],
    ds["bc_flag"].astype(int) - 1,
    input_core_dims=[["bc"], []],
    output_core_dims=[[]],
    vectorize=True,
    dask="allowed",
    output_dtypes=[ds["annualMax"].dtype],
)

ls_messages.append(f'BiasCorrection with bc_flag successfully processed.')

#### Validation

original dimensions are: sample(time), member(ensemble/scenario?), bc, site(location)

In [ ]:
site = 100
sample = 500

In [ ]:
ds.annualMax[sample,:,:,site].values

In [ ]:
ds.annualMax.bc_flag[site].astype(int)-1

In [ ]:
# Assert whether the bias correction works as expected 
# dimensions for ds.annualMax sample(time), member: 2, bc: 3, site-id
# dimensions for annualMax_pref sample(time), member: 2, site-id

expected = ds.annualMax[sample, :, ds.annualMax.bc_flag[site].astype(int)-1, site].values 
actual = annualMax_pref[sample, :, site].values

print(f"expected selection: {expected}... asserting whether actual: {actual} matches...")
assert allclose(actual, expected, equal_nan=True)

### Discarding non_valid data

In [ ]:
data_valid = annualMax_pref.where(ds.model_valid == True, drop=True)

In [ ]:
sites_valid = data_valid.shape[-1]
sites_total = annualMax_pref.shape[-1]
rate_invalid = (1 - sites_valid / sites_total) * 100

ls_messages.append(f'Selection of {sites_valid} valid locations - {rate_invalid:.2f}% invalid sites removed.')
print(
    f"From {sites_total} sites, only {sites_valid} are model valid; "
    f"{rate_invalid:.2f}% invalid sites removed."
    )

# Create pipeline for data import and preparation

In [ ]:
ls_files = [file for file in glob(path + '*.nc')]
ls_files

['../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc']

In [ ]:
def import_data_from_file(file):
    model_name = file.split('/')[-1].split('Annual_max_')[-1].split('.')[0]
    ds = xr.open_dataset(file, engine="netcdf4")
    
    return model_name, ds

def bias_correction(ds):
    return xr.apply_ufunc(
        lambda v, i: v[i], ds["annualMax"], ds["bc_flag"].astype(int) - 1,
        input_core_dims=[["bc"], []], output_core_dims=[[]],
        vectorize=True, dask="allowed", output_dtypes=[ds["annualMax"].dtype],
    )
    
def verify_bias_correction(ds_model, annualMax_pref):
    site = random.randint(0, ds_model.sites.shape[0]-1)
    sample = random.randint(0, ds_model.sample.shape[0]-1)

    expected = ds_model.annualMax[sample, :, ds_model.annualMax.bc_flag[site].astype(int)-1, site].values 
    actual = annualMax_pref[sample, :, site].values

    print(
        f" - Validating bias correction:\n"
        f"\tAsserting whether actual ({actual}) selection matches expected selection ({expected})... "
    )
    assert allclose(actual, expected, equal_nan=True)
    
    
def select_valid_data(ds_model, annualMax_pref):
    data_valid = annualMax_pref.where(ds_model.model_valid == True, drop=True)

    sites_valid = data_valid.shape[-1]
    sites_total = annualMax_pref.shape[-1]
    rate_invalid = (1 - sites_valid / sites_total) * 100

    return data_valid, sites_valid, sites_total, rate_invalid

In [ ]:
for en, file in enumerate(ls_files[:1]):
    
    model_name, ds_model = import_data_from_file(file) 
    print(f'Processing model: {model_name} ({en+1}/{len(ls_files)})...')
    
    ds_model_corrected = bias_correction(ds_model)
    verify_bias_correction(ds_model, ds_model_corrected)
    print(f' - Bias correction successfully done and verified.')

    data_valid, sites_valid, sites_total, rate_invalid = select_valid_data(ds_model, ds_model_corrected)
    print(
        f" - Site Selection done:\n"
        f"\tFrom {sites_total} sites, only {sites_valid} are model valid; {rate_invalid:.2f}% invalid sites removed."
    )

Processing model: MIROC6 (1/8)...
 - Validating bias correction:
	Asserting whether actual ([nan nan]) selection matches expected selection [nan nan]... 
 - Bias correction successfully done and verified.
 - Site Selection done:
	From 11022 sites, only 7054 are model valid; 36.00% invalid sites removed.


# Example Usages

### Extract all annual maxima for a given simulated year (all ensemble members, all lead years)

In [ ]:
year = 1980
am_1980 = annualMax_pref.where(ds.sim_year == year, drop=True)

am_1980

### Extract year + specific lead year

In [ ]:
year = 1980
lead = 3
am_1980_l3 = annualMax_pref.where(
    (ds.sim_year == year) & (ds.lead == lead),
    drop=True
)

am_1980_l3

### Extract time series at one site (all years and leads)

In [ ]:
site_id = 5000
am_site = annualMax_pref.isel(sites=site_id)
ti = am_site['sim_year'].values
lyr = am_site['lead'].values
ami = am_site.values


# Display Model Locations 
Which locations are we looking at? which one are valid?

In [ ]:
df = DataFrame([ds.lon.values, ds.lat.values, ds.model_valid.values], index=['lon', 'lat', 'model_valid']).T
df.info()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_title(
    f"Model {model_name} locations with validity flag · grey: invalid ({sites_total-sites_valid}), "
    f"blue: valid ({sites_valid})", 
    loc='left', fontsize=fs_title
    )

cmap = ListedColormap(["lightgrey", colors[2]])
norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)

ax.scatter(
    df["lon"], df["lat"],
    c=df["model_valid"].astype(int), s=1, cmap=cmap, norm=norm, transform=ccrs.PlateCarree(),
)
ax.coastlines()
ax.set_extent([-22, 40, 25, 65], crs=ccrs.PlateCarree()) 

plt.show()

if save_site_plot:
    export_dir_figures.mkdir(parents=True, exist_ok=True)
    file_name = f"model_{model_name}_site_validity_map.png"
    print(f"saving map to {export_dir_figures} as {file_name}")
    
    fig.savefig(
        export_dir_figures / file_name, dpi=300, bbox_inches='tight'
        )


# Display Timeseries at a specific location

1. Simulation year (init year)
This is the year when the model forecast was initialized.<br>
*Example: sim_year = 1995 means the model started from conditions in 1995.*

2. Lead time
How far into the future the model is predicting from the init year. Often in months or years.<br>
*Example:*
   - lead = 0 → initial state (or first forecast period)
   - lead = 1 → 1 year (or 1 month) after initialization
   - lead = 5 → 5 years after initialization

1. The actual time represented by a data point
The valid (verification) time is:
$valid year = sim year + lead$


**Questions to look at**<br>
A. *“What did the 1990 forecast predict over time?”* → Fix init year → see forecast evolution<br>
B. *“How good are 3-year lead forecasts across all hindcasts?”* → Fix lead time → compare across init years

In [ ]:
location_lookup = input(f"Inspect the closest value to a specific location ")

In [ ]:
location_lookup == ''

In [ ]:
if location_lookup == 'None' or location_lookup == '':
    print(f"No specific location selected.")
    location_closest = None
else:
    message = f'find closest location to {location_lookup}...'
    print(message)
    ls_messages.append(message)
    
    location_closest = geolocator.geocode(location_lookup)
    location_closest.latitude, location_closest.longitude

In [ ]:
def find_closest_site_xr(sites_da, target_lat, target_lon):
    dist2 = (sites_da.lat - target_lat)**2 + (sites_da.lon - target_lon)**2
    
    idx_min = dist2.argmin().item()
    
    return {
        "index": idx_min,
        "site_id": sites_da.sites[idx_min].item(),
        "lat": sites_da.lat[idx_min].item(),
        "lon": sites_da.lon[idx_min].item()
    }

if location_closest:
    site_found = find_closest_site_xr(data_valid.sites, location_closest.latitude, location_closest.longitude)

    site_id = site_found['index']
    message = f'closest location found: {site_id}, {geolocator.reverse((site_found["lat"], site_found["lon"]))}'
    ls_messages.append(message)
    print(message)
else:
    site_id = 300 #random.randint(0, data_valid.sites.sites.shape[0]-1)
    
site_id

In [ ]:
am_site = data_valid.isel(sites=site_id) # data_valid[:, :, site_id] - sample, member, annualMax

sim_years = am_site.sim_year.to_series().unique().astype(int)
lead = am_site.lead.to_series().unique()

In [ ]:
try:
    location = geolocator.reverse((am_site.lat.values, am_site.lon.values))
except: 
    location = None
location

In [ ]:
df_site = DataFrame(am_site.values, columns=['annualMax_member0', 'annualMax_member1'])
df_site['sim_year'] = DataFrame(am_site['sim_year'].values.astype(int))
df_site['lead'] = DataFrame(am_site['lead'].values.astype(int))
df_site['valid_year'] = df_site.sim_year + df_site.lead


message = f" - fetched annual maximum storm-surge with {df_site.dropna().shape[0]} valid samples out of {df_site.shape[0]} total samples."
ls_messages.append(message)
print(message)

df_site.dropna()

### A. “What did the *sim_year* forecast predict over time?”
Fix init year → see forecast evolution<br>

In [ ]:
sim_years

In [ ]:
sim_year_selected = random.choice(sim_years)
sim_year_selected

In [ ]:
def select_location_data_with_inityear(df_site: DataFrame, sim_year_selected: int) -> DataFrame:
    return df_site[df_site.sim_year == sim_year_selected].sort_values('valid_year')

# ----------------------------------------------------

sim_year_forecast = select_location_data_with_inityear(df_site, sim_year_selected)
sim_year_forecast

In [ ]:
def plot_forecast_timeseries_from_inityear_for_location( 
    sim_year_forecast: DataFrame, sim_year_selected: int, location: Location, site_id:int, model_name: str,
    save_time_series_plot: bool = True,  figsize=(13, 3.5), 
    ) -> plt.Figure:
    
    fig, ax = plt.subplots(figsize=figsize)
    title = "Annual storm-surge maxima at "
    title += f"{location.address} (site-id {site_id})" if location else f"site-id {site_id}"
    ax.set_title(
        title + f" for simulation year {sim_year_selected} (Overview, model {model_name})", 
        loc='left', fontsize=fs_title
        )

    sim_year_forecast.set_index('valid_year').filter(like='annualMax').plot(ax=ax, marker='o')

    ax.set_ylabel('Annual storm-surge maxima, m')
    ax.set_xlabel('verification year')

    ax.grid(False)
    sns.despine()
    plt.tight_layout()

    if save_time_series_plot:
        export_dir_figures.mkdir(parents=True, exist_ok=True)
        file_name = f"sim_year/forecast_with_simyear-{sim_year_selected}_for_{location}_model_{model_name}.png"
        print(f"saving map to {export_dir_figures/"sim_year"} as {file_name}")
        
        fig.savefig(
            export_dir_figures / file_name, dpi=300, bbox_inches='tight'
            )

    plt.close(fig)
    return fig

# ----------------------------------------------------

fig = plot_forecast_timeseries_from_inityear_for_location(
    sim_year_forecast, sim_year_selected, location, site_id, model_name,save_time_series_plot=True
    )
fig
    

#### Full run through all options

In [ ]:
years_selected_for_comparison = [1966, 1988, 1999, 2006, 2010, 2021, 2024, 2026, 2028]

In [ ]:
for sim_year_selected in years_selected_for_comparison: 
    sim_year_forecast = select_location_data_with_inityear(df_site, int(sim_year_selected))

    fig = plot_forecast_timeseries_from_inityear_for_location(
        sim_year_forecast, sim_year_selected, location, site_id, model_name,save_time_series_plot=True
        )


### B. “How good are *x-year lead forecasts* across all hindcasts?”
Fix lead time → compare across init years

In [ ]:
lead_selected = random.choice(lead).astype(int)
lead_selected = 3

In [ ]:
def select_location_data_with_lead(df_site: DataFrame, lead_selected: int) -> DataFrame:
    lead_year_forecast = df_site[df_site.lead == lead_selected]
    return lead_year_forecast.sort_values('valid_year')

# ----------------------------------------------------

lead_year_forecast = select_location_data_with_lead(df_site, lead_selected)
lead_year_forecast

In [ ]:
def plot_forecast_quality_at_location_for_leadyear(
    lead_year_forecast: DataFrame, lead_selected: int, location: Location, site_id:int, model_name: str, 
    save_time_series_plot: bool = True, figsize=(13, 3.5), 
    ) -> plt.Figure:
    
    fig, ax = plt.subplots(figsize=figsize)
    
    title = "Annual storm-surge maxima at "
    title += f"{location.address} (site-id {site_id})" if location else f"site-id {site_id}"
    ax.set_title(
        title + f" for lead year {lead_selected} (Overview, model {model_name})", 
        loc='left', fontsize=fs_title
        )

    lead_year_forecast.set_index('valid_year').filter(like='annualMax').plot(ax=ax, marker='o')

    ax.set_ylabel('Annual storm-surge maxima, m')
    ax.set_xlabel('verification year')

    ax.grid(False)
    sns.despine()
    plt.tight_layout()

    if save_time_series_plot:
        export_dir_figures.mkdir(parents=True, exist_ok=True)
        file_name = f"lead/forecastQuality_at_leadyear-{lead_selected}_for_{location}_model_{model_name}.png"
        print(f"saving map to {export_dir_figures} as: {file_name}")
        
        fig.savefig(
            export_dir_figures / file_name, dpi=300, bbox_inches='tight'
            )
    plt.close(fig)
    return fig

# ----------------------------------------------------

fig = plot_forecast_quality_at_location_for_leadyear(
    lead_year_forecast, lead_selected, location, site_id, model_name, save_time_series_plot=True
    )

fig

#### Full run through all options

In [ ]:
for lead_selected in lead: 
    lead_year_forecast = select_location_data_with_lead(df_site, int(lead_selected))

    fig = plot_forecast_quality_at_location_for_leadyear(
        lead_year_forecast, int(lead_selected), location, site_id, model_name, save_time_series_plot=True
        )
